# Notes on computing pokemon stats

**<u>Basic Stats:</u>**
1. HP
2. Attack ($A$)
3. Defense ($D$)
4. Speed ($S$)
5. Special Attack ($S_A$)
6. Special Defense ($S_D$)

**<u>Stat Parameters</u>**  
Computing the value of any of the above stats requires additional parameters. These differ for each Pokemon *and each stat* $X$:
1. $B=B_{X} \in [0..255]$ is the *Base Value* (BV); these are immutable, fixed by Nintendo for each pokemon in each game.
2. $E=E_X \in [0..255]$ is the *Effort Value* (EV); to be given by Team Generator
3. $I=I_X \in [0..31]$ is the *Individual Value* (IV); to be given by Team Generator
4. $N=N_X \in \{0.9,1.,1.1\}$ is the *Nature Multiplier* (my term); more on that in a bit.

**<u>Computing actual stat values</u>**  
Fix a Pokemon $P$, let $L$ be its **Level**, and let $N$ be its **Nature**. For each stat $X$, let 
    $$ Q = Q_X := \Big({2B+I+\lfloor \tfrac{E}{4} \rfloor}\Big)\frac{L}{100} \qquad (B=B_X,\,\text{etc.})$$
Then $\mathrm{HP}$ is computed via
    $$ \mathrm{Value}(\mathrm{HP}) = \lfloor Q \rfloor + L + 10, $$
and the remaining five stats (Attack, Defense, etc.) are computed via
    $$ \mathrm{Value}(X) = \big\lfloor (Q_{X}+5)N_{X} \big\rfloor. $$

**<u>Nature Multipliers</u>**
<span style="color:red">Update/Note:</span> Ostensibly the `Nature` attribute does not feature in `gen9randombattle`?  
Natures are just "categories/types" (e.g. *Adamant*, *Hardy*) that *raise* one non-HP stat $10\%$ and *lower* one non-HP stat $10\%$; some actually "do nothing" because they both "raise and lower" the same stat. Thus, we have the following table of multipliers for each Nature;  

(For easier reading, "." means $1$) 

| (Nat\Stat) | Atk | Def | SpA | SpD | Spe |
|---|:---:|:---:|:---:|:---:|:---:|
| Adamant | 1.1 | . | 0.9 | . | . |
| Bashful | . | . | . | . | . |
| Bold | 0.9 | 1.1 | . | . | . |
| Brave | 1.1 | . | . | . | 0.9 |
| Calm | 0.9 | . | . | 1.1 | . |
| Careful | . | . | 0.9 | 1.1 | . |
| Docile | . | . | . | . | . |
| Gentle | . | 0.9 | . | 1.1 | . | 
| Hardy | . | . | . | . | . |
| Hasty | . | 0.9 | . | . | 1.1 | 
| Impish | . | 1.1 | 0.9 | . | . | 
| Jolly | . | . | 0.9 | . | 1.1 | 
| Lax | . | 1.1 | . | 0.9 | . | 
| Lonely | 1.1 | 0.9 | . | . | . |
| Mild | . | 0.9 | 1.1 | . | . | 
| Modest | 0.9 | . | 1.1 | . | . | 
| Naive | . | . | . | 0.9 | 1.1 |
| Naughty | 1.1 | . | . | 0.9 | . | 
| Quiet | . | . | 1.1 | . | 0.9 | 
| Quirky | . | . | . | . | . |
| Rash | . | . | 1.1 | 0.9 | . | 
| Relaxed | . | 1.1 | . | . | 0.9 | 
| Sassy | . | . | . | 1.1 | 0.9 | 
| Serious | . | . | . | . | . |
| Timid | 0.9 | . | . | . | 1.1 | 

In [27]:
NAT_MULTS = {
    "Adamant": [1.1, 1., 0.9, 1., 1.],
    "Bashful": [1., 1., 1., 1., 1.],
    "Bold":    [0.9, 1.1, 1., 1., 1.],
    "Brave":   [1.1, 1., 1., 1., 0.9],
    "Calm":    [0.9, 1., 1., 1.1, 1.],
    "Careful": [1., 1., 0.9, 1.1, 1.],
    "Docile":  [1., 1., 1., 1., 1.],
    "Gentle":  [1., 0.9, 1., 1.1, 1.],
    "Hardy":   [1., 1., 1., 1., 1.],
    "Hasty":   [1., 0.9, 1., 1., 1.1],
    "Impish":  [1., 1.1, 0.9, 1., 1.],
    "Jolly":   [1., 1., 0.9, 1., 1.1],
    "Lax":     [1., 1.1, 1., 0.9, 1.],
    "Lonely":  [1.1, 0.9, 1., 1., 1.],
    "Mild":    [1., 0.9, 1.1, 1., 1.],
    "Modest":  [0.9, 1., 1.1, 1., 1.],
    "Naive":   [1., 1., 1., 0.9, 1.1],
    "Naughty": [1.1, 1., 1., 0.9, 1.],
    "Quiet":   [1., 1., 1.1, 1., 0.9],
    "Quirky":  [1., 1., 1., 1., 1.],
    "Rash":    [1., 1., 1.1, 0.9, 1.],
    "Relaxed": [1., 1.1, 1., 1., 0.9],
    "Sassy":   [1., 1., 1., 1.1, 0.9],
    "Serious": [1., 1., 1., 1., 1.],
    "Timid":   [0.9, 1., 1., 1., 1.1],
}

# Implementing
Steps
1. Get `player_dets` from `replay_file`
2. Feed `player['seed']` to `get_team` to get `Team`(s) 
3. Pass `seed` to 'server' to get `Team`
4. Get stats from Pokedex and `Team`
5. Compute actual `stats` for each pokemon using `statD`
--------

### Getting Player Details

In [46]:
import copy, json, re, time
import itertools, requests
import logging

import numpy as np

from pathlib import Path
from urllib.parse import urlencode, urljoin

In [48]:
# First, get 'teams_full' array: [<dict1>, <dict2>], with <dict#> = { <pokename> : <dict_of_info> }
# ===========================
REPLAY_DIR = Path('./../data/replays/gen9-randombattle')

with open('./pokedex.json', 'r') as file:
    POKEDEX = json.load(file)

# replay = REPLAY_DIR / 'gen9randombattle-2631360263.json' (for test)

for replay in REPLAY_DIR.iterdir():
    if (replay.is_file() and replay.name[0]!='.'): # skipping hidden files
        try:
            with replay.open() as file:
                battle_json = json.load(file)
            TEAMS = get_teams_full(battle_json)
    
            for team in TEAMS:
                append_team_stats(team)
            
            battle_json['teams_full'] = TEAMS
            replay.write_text(json.dumps(battle_json), encoding='utf-8')
        
        except:
            print("Could not open file: %s" % replay.name)
            continue
    else:
        print("Could not open file: %s" % replay.name)
        continue

Could not open file: .DS_Store
Could not open file: .ipynb_checkpoints


In [35]:
# ===============================================
# Macro functions
# ===============================================
def get_teams_full(battle_json: dict):
    '''
    Intake a replay json, compute its full teams and stats.
    '''
    
    # (1) Add 'player_dets' entry
    battle_json['player_dets'] = get_player_dets(battle_json)

    # (2)-(3) Get `seed`s and pass these to "server" to receive full team list.
    _temp_array = []
    
    for player in battle_json['player_dets']:
        seed = player['seed'] 
        _temp_array.append(get_team(seed))

    # Now we have convert _temp_array -> teams_full: 
    # _temp_array[i] = [{poke_i1}, ..., {poke_i6}]
    #     |-> teams_full[i] = ['poke_i1':{poke_i1}, ..., 'poke_i6':{poke_i6}]
    teams_full = []
    for team in _temp_array:
        team_D = dict()
        for poke in team:
            team_D[poke['name']] = copy.deepcopy(poke)
        teams_full.append(team_D)

    return teams_full
# ===========================

# add the BVs and computed Stats to each Pokemon 
def append_team_stats(team_D: dict):
    '''
    Input poke dict, compute total stats for each Pokemon in it, append to Pokemon, and return team.
    '''
    for name in team_D.keys():
        poke = team_D[name] # get the `dict`

        # (4)-(5)
        poke['bvs'] = copy.deepcopy(POKEDEX[poke['speciesId']]['baseStats']) # deepcopy for safety
        poke['stats'] = compute_stats(poke)

    return None
    

In [25]:
# ===============================================
# Intermediate functions
# ===============================================
def get_player_dets(battle_json):
    try:
        match1 = re.search(r'\>player p1 ({.*?})$', battle_json['inputlog'], re.M)
        p1_json = json.loads(match1.group(1))
        assert p1_json['name'] == battle_json['players'][0] # just to be certain
    except:
        print("Couldn't get p1 info.")
        return None

    try:
        match2 = re.search(r'\>player p2 ({.*?})$', battle_json['inputlog'], re.M)
        p2_json = json.loads(match2.group(1))
        assert p2_json['name'] == battle_json['players'][1] # just to be certain
    except:
        print("Couldn't get p2 info.")
        return None
    
    return [p1_json, p2_json]

# ===========================
def get_team(seed):
    params = urlencode({
            "format": 'gen9randombattle', 
            "seed": seed
        }) 
    url = f"http://localhost:3000?{params}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.RequestsException as e: 
        print("Could not get team: %s", e)
        return []
    
    try: 
        team = json.loads(response.content.decode())
    except json.JSONDecodeError as e:
        logger.error("failed to parse team: %s", e)
        return []
        
    return team

# ===========================
# the computation function
def compute_stats(poke: dict) :
    '''
    `poke` should have dictionaries `BV`, `EV`, `IV`, and entry `level`
    example output: {'hp': 263, 'atk': 120, 'def': 169, 'spa': 240, 'spd': 216, 'spe': 223}
    '''
    _stat_D = {}

    BV = poke['bvs']
    EV = poke['evs']
    IV = poke['ivs']
    lvl = poke['level']

    for k in BV.keys():
        Q_k = (2*BV[k] + IV[k] + np.floor(EV[k]/4))*(lvl/100)
        nat_k = 1.0 # in case we want to incorporate `nature`s later
        
        if k == 'hp':
            _stat_D[k] = int(np.floor(Q_k) + lvl + 10)
        else:
            _stat_D[k] = int(np.floor((Q_k+5)*nat_k))

    return _stat_D